### LangGraph 로깅

프로그램 실행 중 발생한 일을 남긴 기록을 로그(log), 기록하는 작업을 로깅(logging)이라고 한다. 로그에 시각, 실행 단계, 처리 결과를 남기면 실행 흐름과 오류 원인을 확인할 수 있다.

Python의 `logging`은 기본 모듈이므로 별도 설치 없이 사용할 수 있다.

### print() 대신 logging을 쓰는 이유

`print()`로 실행 과정을 확인할 수 있지만, 출력이 많아지면 필요한 기록만 골라 보기 어렵다. `logging`을 사용하면 설정만으로 기록할 중요도, 출력 형식, 저장 위치를 조절할 수 있다.

### 로거, 핸들러, 포매터

- **로거(Logger)**: `logger.info()`처럼 로그를 기록하는 객체다.
- **핸들러(Handler)**: 로그를 화면이나 파일 등 지정한 위치로 보낸다. 하나의 로거에 여러 핸들러를 연결할 수 있다.
- **포매터(Formatter)**: 시각, 레벨, 메시지 등 로그의 표시 형식을 정한다. 핸들러에 연결해 사용한다.


### 설정 코드

- `getLogger("lesson.langgraph")`: 이 이름의 로거를 가져온다. 같은 이름으로 호출하면 같은 로거를 얻는다.
- `setLevel(logging.INFO)`: INFO 이상의 중요도를 기록한다.
- `StreamHandler(sys.stdout)`: 노트북 출력 영역으로 보낸다.
- `setFormatter(...)`: 로그의 표시 형식을 정한다.
- `addHandler(...)`: 로거에 출력 목적지를 연결한다.
- `propagate = False`: 상위 로거로 다시 전달하지 않는다. 이 예제에서는 직접 연결한 핸들러만 사용한다.

`logger.handlers[:]`는 연결된 핸들러 목록의 복사본이다. 기존 핸들러를 제거하고 파일 등 사용 중인 자원을 닫은 뒤 새로 연결한다.

`.py` 파일에서는 보통 `logging.getLogger(__name__)`으로 모듈 이름을 사용한다.

In [1]:
import logging
import sys

logger = logging.getLogger("lesson.langgraph")
logger.setLevel(logging.INFO)
logger.propagate = False

# 설정 셀을 다시 실행해도 같은 로그가 중복 출력되지 않게 한다.
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
    handler.close()

console_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

logger.info("로깅 준비 완료")

09:18:55 | INFO | lesson.langgraph | 로깅 준비 완료


### 출력 확인

출력 예시에서 시각은 실행할 때마다 달라진다.

```text
14:30:00 | INFO | lesson.langgraph | 로깅 준비 완료
```

`%(asctime)s`는 시각, `%(levelname)s`는 레벨, `%(name)s`는 로거 이름, `%(message)s`는 메시지다.

### 로그 레벨로 중요도 나누기

| 레벨 | 기록할 상황 |
|---|---|
| `DEBUG` | 검색 결과 개수 등 자세한 진단 정보 |
| `INFO` | 노드 시작, 완료, 선택한 경로 |
| `WARNING` | 검색 결과가 없어 안내 답변으로 전환 |
| `ERROR` | 외부 서비스 오류로 요청 처리 실패 |
| `CRITICAL` | 서비스 전체를 계속 운영하기 어려운 심각한 문제 |

중요도는 `DEBUG < INFO < WARNING < ERROR < CRITICAL` 순서다. 레벨을 INFO로 정하면 DEBUG만 걸러진다. WARNING으로 정하면 WARNING, ERROR, CRITICAL이 남는다.

설정하지 않은 기본 루트 로거의 기준은 WARNING이다. 앞의 코드에서는 로거의 레벨을 INFO로 지정했다. 아래 ERROR와 CRITICAL은 출력 연습용 메시지이며 실제 오류를 발생시키지는 않는다.


In [ ]:
logger.debug("연습: 검색 문서 개수 확인")
logger.info("연습: 요청 처리 시작")
logger.warning("연습: 검색 결과 없음")
logger.error("연습: 요청 처리 실패")
logger.critical("연습: 서비스 운영 불가")

In [ ]:
logger.setLevel(logging.WARNING)
logger.info("이 INFO는 보이지 않는다")
logger.warning("이 WARNING은 보인다")

logger.setLevel(logging.DEBUG)
logger.debug("이제 DEBUG도 보인다")

logger.setLevel(logging.INFO)

### 변수와 함께 기록하기

f-string으로 변수 값을 로그에 포함할 수 있다.

In [ ]:
node_name = "search"
document_count = 2
elapsed_ms = 12.345

logger.info(f"node={node_name} documents={document_count} elapsed_ms={elapsed_ms:.2f}")

### 화면과 파일에 함께 남기기

화면을 닫은 뒤에도 살펴볼 수 있도록 파일 핸들러를 추가한다. `Path.cwd()`는 현재 작업 폴더다. 그 아래 `logs/29-langgraph.log`가 만들어진다.

`mode="a"`는 기존 기록 뒤에 추가한다. `encoding="utf-8"`은 한글 저장에 사용한다. 설정 셀을 다시 실행할 때는 파일 핸들러를 교체해 중복 저장을 막는다.

In [ ]:
from pathlib import Path

log_dir = Path.cwd() / "logs"
log_dir.mkdir(exist_ok=True)
log_path = log_dir / "29-langgraph.log"

for handler in logger.handlers[:]:
    if isinstance(handler, logging.FileHandler):
        logger.removeHandler(handler)
        handler.close()

file_handler = logging.FileHandler(log_path, mode="a", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("화면과 파일에 함께 기록")
print("로그 파일:", log_path.resolve())

### 화면은 간단하게, 파일은 자세하게

로그는 **로거의 기준을 통과한 뒤, 각 핸들러의 기준도 통과해야** 출력된다. 로거가 INFO라면 파일 핸들러만 DEBUG로 바꿔도 DEBUG 기록은 남지 않는다.

여기서는 로거를 DEBUG, 화면을 INFO, 파일을 DEBUG로 정한다. 이후 DEBUG 기록은 파일에서만 확인한다.

In [ ]:
logger.setLevel(logging.DEBUG)
console_handler.setLevel(logging.INFO)
file_handler.setLevel(logging.DEBUG)

logger.debug("파일에서만 보이는 상세 기록")
logger.info("화면과 파일에 모두 보이는 진행 기록")

file_handler.flush()
print(log_path.read_text(encoding="utf-8"))

### 로그 파일 로테이션

로그를 계속 추가하면 파일이 커진다. `RotatingFileHandler`는 파일 크기를 기준으로 새 파일로 전환하고, 이전 파일은 정해진 개수만 보관한다.

- `maxBytes`: 파일을 교체할 크기 기준(바이트).
- `backupCount`: 보관할 이전 파일 개수. 현재 기록 중인 파일은 제외한다.

아래 코드는 기존 파일 핸들러를 교체한다. 회전을 확인하기 위해 크기 기준을 1,024바이트로 작게 설정하고 로그를 반복해서 남긴다.

현재 기록은 `29-langgraph.log`, 이전 기록은 `29-langgraph.log.1`, `29-langgraph.log.2`에 저장된다. `.1`이 더 최근 기록이며, 보관 개수를 넘으면 가장 오래된 파일이 삭제된다.



In [ ]:
from logging.handlers import RotatingFileHandler

logger.removeHandler(file_handler)
file_handler.close()

file_handler = RotatingFileHandler(
    log_path,
    maxBytes=1024,
    backupCount=2,
    encoding="utf-8",
)
file_handler.setFormatter(formatter)
file_handler.setLevel(logging.DEBUG)
logger.addHandler(file_handler)

for index in range(40):
    logger.debug(f"파일 회전 확인: 반복 기록 {index}")

file_handler.flush()
for path in sorted(log_dir.glob("29-langgraph.log*")):
    print(path.name)


### 오류 메시지와 traceback 기록하기

`logger.error()`는 기본적으로 메시지만 남긴다. `except` 안의 `logger.exception()`은 ERROR 레벨 메시지와 현재 예외의 traceback을 함께 남긴다. Traceback은 어떤 함수와 코드 위치를 거쳐 오류가 발생했는지 보여 준다.

**기록하는 것과 오류를 처리하는 것은 별개다.** `logger.exception()`만으로 예외가 다시 발생하거나 작업이 재시도되지 않는다. 호출한 곳에 실패를 전달하려면 `raise`를 사용한다.

In [ ]:
try:
    int("숫자가 아님")
except ValueError:
    logger.error("정수 변환 실패: 메시지만 기록")
    logger.exception("정수 변환 실패: traceback도 기록")

### LangGraph 노드에 로그 남기기

노드도 Python 함수이므로 앞에서 만든 로거를 그대로 사용한다. 질문의 공백을 제거하는 `prepare`와 답변을 만드는 `answer`를 연결하고, 각 노드의 시작과 완료를 기록한다.

```text
START → prepare → answer → END
```

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class LoggingState(TypedDict):
    question: str
    answer: str


def prepare(state: LoggingState):
    logger.info("prepare 시작")
    question = state["question"].strip()
    logger.info("prepare 완료")
    return {"question": question}


def answer(state: LoggingState):
    logger.info("answer 시작")
    try:
        if not state["question"]:
            raise ValueError("질문이 비어 있습니다.")
        response = f"질문을 받았습니다: {state['question']}"
    except ValueError:
        logger.exception("answer 실패")
        raise
    logger.info("answer 완료")
    return {"answer": response}


builder = StateGraph(LoggingState)
builder.add_node("prepare", prepare)
builder.add_node("answer", answer)
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "answer")
builder.add_edge("answer", END)
graph = builder.compile()

### 정상 실행 확인하기

`prepare 시작 → prepare 완료 → answer 시작 → answer 완료` 순서로 INFO 로그가 남는다.

In [ ]:
result = graph.invoke({"question": "  LangGraph가 무엇인가요?  "})
print(result["answer"])

### 오류 위치 확인하기

공백만 입력하면 `answer`에서 `ValueError`가 발생한다. `logger.exception()`은 실패 메시지와 traceback을 남기고, `raise`는 예외를 호출한 곳으로 전달한다. 로그에 `answer 시작`과 `answer 실패`는 있지만 `answer 완료`는 없다.

아래 `except`는 실습 중 발생시킨 예외를 잡는다.

In [ ]:
try:
    graph.invoke({"question": "   "})
except ValueError as error:
    print("실행 실패:", error)